# **Computational Drug Discovery [Part 1] Download Bioactivity Data**


## **ChEMBL Database**

The [*ChEMBL Database*](https://www.ebi.ac.uk/chembl/) is a database that contains curated bioactivity data of more than 2 million compounds. It is compiled from more than 76,000 documents, 1.2 million assays and the data spans 13,000 targets and 1,800 cells and 33,000 indications.
[Data as of March 25, 2020; ChEMBL version 26].

## **Installing libraries**

Install the ChEMBL web service package so that we can retrieve bioactivity data from the ChEMBL Database.

In [5]:
! pip install chembl_webresource_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 5.9 MB/s eta 0:00:00


## **Importing libraries**

In [6]:
# Import necessary libraries
import pandas as pd
from chembl_webresource_client.new_client import new_client

/usr/local/lib/python3.13/dist-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


## **Search for Target protein**

### **Target search for EGFR**

In [7]:

target = new_client.target
target_query = target.search('EGFR')
targets = pd.DataFrame.from_dict(target_query)
targets

,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Homo sapiens,EGFR/PPP1CA,17.0,False,CHEMBL4523747,"[{'accession': 'P00533', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606
1,[],Homo sapiens,CCN2-EGFR,17.0,False,CHEMBL5465557,"[{'accession': 'P00533', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606
2,[],Mus musculus,Epidermal growth factor receptor,15.0,False,CHEMBL3608,"[{'accession': 'Q01279', 'component_descriptio...",SINGLE PROTEIN,10090
3,[],Mus musculus,Protein cereblon/Epidermal growth factor receptor,13.0,False,CHEMBL6193842,"[{'accession': 'Q01279', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,10090
4,[],Homo sapiens,Epidermal growth factor receptor,11.0,False,CHEMBL203,"[{'accession': 'P00533', 'component_descriptio...",SINGLE PROTEIN,9606
5,[],Homo sapiens,Protein cereblon/Epidermal growth factor receptor,11.0,False,CHEMBL4523680,"[{'accession': 'P00533', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606
6,[],Homo sapiens,Epidermal growth factor receptor,10.0,False,CHEMBL2363049,"[{'accession': 'P04626', 'component_descriptio...",PROTEIN FAMILY,9606
7,[],Homo sapiens,MER intracellular domain/EGFR extracellular do...,10.0,False,CHEMBL3137284,"[{'accession': 'P00533', 'component_descriptio...",CHIMERIC PROTEIN,9606
8,[],Homo sapiens,von Hippel-Lindau disease tumor suppressor/Epi...,10.0,False,CHEMBL4523998,"[{'accession': 'P00533', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606
9,[],Mus musculus,Protein cereblon/Epidermal growth factor receptor,10.0,False,CHEMBL6193841,"[{'accession': 'P00533', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,10090


In [8]:
selected_target = targets.target_chembl_id[4]
selected_target

'CHEMBL203'

In [ ]:
import os
import pandas as pd
from chembl_webresource_client.new_client import new_client

OUTPUT_DIR = '' # Define OUTPUT_DIR. Set to 'data/' or similar if you want a subfolder.
RAW_CSV = OUTPUT_DIR + 'egfr_raw_chembl.csv'

if os.path.exists(RAW_CSV):
    print('Loading saved ChEMBL data...')
    df_raw = pd.read_csv(RAW_CSV)
    print(f'Loaded {len(df_raw)} records from cache.')
else:
    print('Querying ChEMBL (this may take a minute)...')
    activity = new_client.activity
    egfr_raw = list(activity.filter(
        target_chembl_id       = 'CHEMBL203',
        standard_type          = 'IC50',
        standard_units         = 'nM',
        standard_relation      = '=',
        assay_confidence_score = 9,
        pchembl_value__isnull  = False,
        canonical_smiles__isnull = False,
    ).only([
        'molecule_chembl_id',
        'canonical_smiles',
        'standard_value',
        'standard_units',
    ])[:10000])

    df_raw = pd.DataFrame(egfr_raw)
    df_raw['standard_value'] = pd.to_numeric(
        df_raw['standard_value'], errors='coerce')
    df_raw = df_raw.rename(columns={'standard_value': 'IC50_nM'})
    df_raw = df_raw.dropna(subset=['canonical_smiles', 'IC50_nM'])
    df_raw = df_raw[df_raw['IC50_nM'] > 0]
    df_raw.to_csv(RAW_CSV, index=False)
    print(f'Query complete. {len(df_raw)} records saved to {RAW_CSV}')

print(f'Unique compounds: {df_raw["molecule_chembl_id"].nunique()}')
df_raw.head()


Querying ChEMBL (this may take a minute)...


In [ ]:
import numpy as np

def geometric_mean_ic50(values):
    """Geometric mean of a list of positive IC50 values."""
    vals = [v for v in values if v > 0]
    if not vals:
        return np.nan
    return float(np.exp(np.mean(np.log(vals))))


df_dedup = (
    df_raw.groupby('molecule_chembl_id')
    .agg(
        canonical_smiles=('canonical_smiles', 'first'),
        IC50_nM_list=('IC50_nM', list),
    )
    .reset_index()
)
df_dedup['IC50_nM'] = df_dedup['IC50_nM_list'].apply(geometric_mean_ic50)
df_dedup = df_dedup.drop(columns='IC50_nM_list')
df_dedup = df_dedup.dropna(subset=['IC50_nM'])

print(f'After deduplication: {len(df_dedup)} unique compounds')
print(f'IC50 range: {df_dedup["IC50_nM"].min():.2f} - {df_dedup["IC50_nM"].max():.0f} nM')


Labeling compounds as either being active, inactive or intermediate
The bioactivity data is in the IC50 unit. Compounds having values of less than 1000 nM will be considered to be active while those greater than 10,000 nM will be considered to be inactive. As for those values in between 1,000 and 10,000 nM will be referred to as intermediate.


In [ ]:
bioactivity_class = []
for i in df_dedup.IC50_nM:
  if float(i) >= 10000:
    bioactivity_class.append("inactive")
  elif float(i) <= 1000:
    bioactivity_class.append("active")
  else:
    bioactivity_class.append("intermediate")

In [ ]:
data_tuples = list(zip(df_dedup['molecule_chembl_id'], df_dedup['canonical_smiles'], bioactivity_class, df_dedup['IC50_nM']))
df = pd.DataFrame( data_tuples,  columns=['molecule_chembl_id', 'canonical_smiles', 'bioactivity_class', 'IC50_nM'])

In [ ]:
df_dedup.to_csv('dedup_preprocessed_data.csv', index=False)

In [ ]:
import sys
if 'rdkit' not in sys.modules:
  !pip install rdkit

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

def compute_descriptors(mol):
    """Compute a standard set of physicochemical descriptors."""
    return {
        'MW': round(Descriptors.MolWt(mol), 2),
        'LogP': round(Descriptors.MolLogP(mol), 2),
        'HBD': rdMolDescriptors.CalcNumHBD(mol),
        'HBA': rdMolDescriptors.CalcNumHBA(mol),
        'RotB': rdMolDescriptors.CalcNumRotatableBonds(mol),
        'TPSA': round(rdMolDescriptors.CalcTPSA(mol), 2),
        'Rings': rdMolDescriptors.CalcNumRings(mol),
        'ArRings': rdMolDescriptors.CalcNumAromaticRings(mol),
    }

In [ ]:
import numpy as np
from rdkit import Chem

# Parse SMILES
df_dedup["mol"] = df_dedup["canonical_smiles"].apply(Chem.MolFromSmiles)
n_before = len(df_dedup)
df_dedup = df_dedup[df_dedup["mol"].notna()].copy()
print(f"Valid structures: {len(df_dedup)} / {n_before}")

In [ ]:
# Descriptors
print("Computing descriptors...")
desc_rows = [compute_descriptors(mol) for mol in df_dedup["mol"]]
desc_df = pd.DataFrame(desc_rows, index=df_dedup.index)
for col in desc_df.columns:
    df_dedup[col] = desc_df[col]

In [ ]:
df_dedup["log_IC50"] = np.log10(df_dedup["IC50_nM"])